# Anest_start — Hb analysis after Step4 (version 2)

**The statistical unit is the mouse, not a frame or brain pixel.** For each mouse, the notebook reads the processed `dcHbT`, `dcHHb`, and `dcHbO` arrays and the Step2 brain mask (`brain_manual.npy` preferred over `brain_auto.npy`).

1. **Units:** by default, numeric Zarr concentrations are assumed to be in **M (mol/L)** and are multiplied by **10⁶** for display in **µM** after extracting the trace. This is an explicit notebook setting, not inference from YAML. The unit metadata are checked and a conflict stops processing by default.
2. **Baseline:** subtract each pixel's mean over the first **5 seconds**, then spatially average the baseline-subtracted values across the brain mask for each frame.
3. **Windows:** the first **7 seconds** and last **60 seconds** of each recording. For each mouse and Hb component, calculate **Δ = last-60-s mean − first-7-s mean** in µM.
4. **Statistics:** a two-sided, one-sample t-test of mouse-level Δ values versus zero; **Holm correction** across the three Hb components within each group. Report n, mean ± SD, 95% confidence interval, and p-values. Wilcoxon is exploratory only.
5. **Figures:** mouse-level Δ scatter and boxplots, group mean ± SD traces, and separate panels of individual mouse traces with their group mean and SD. Colors: HbT dark burgundy, HHb dark blue, and HbO muted red.

**Interactive mouse selection:** Zarr data are read during the preparation stage; the subsequent `ipywidgets` panel recalculates tables, statistics, and plots **from in-memory records**, without reloading the Zarr stores. Original Zarr and Step2 masks are not modified.

By default, every discovered mouse belongs to the group whose literal data label is `Группа 1`. Multiple groups may be specified in the settings cell. If Step4 concentrations are already stored in µM, set `SOURCE_HB_UNITS` accordingly **before processing** to avoid multiplying twice.


In [ ]:
from pathlib import Path

# Paths
CONFIG_PATH = Path(r"config_anest_start.yaml")
PROJECT_ROOT_OVERRIDE = Path(r"Main_analysis_Anest_start_yyyy-dd-mm")  # Example: Path(r"F:\PT2021_final_analysis\Main_analysis_Anest_start_yyyy-dd-mm")
OUTPUT_DIR_OVERRIDE = None    # None -> <project>/out/step4_raw_and_signals/anest_start_summary

# Zarr variables
SIGNAL_SUFFIX = ""            # "" (dcHbT, dcHHb, dcHbO) or "_vreg" / "_gsub"
SOURCE_HB_UNITS = "M"         # "M" -> multiply by 1e6; "µM" -> leave unchanged.
ALLOW_METADATA_CONFLICT = False  # True ONLY when incorrect metadata have been independently verified.

# Groups (None selects all recordings; appropriate only for a single group).
GROUPS = {"Группа 1": None}
# These mice start unchecked in the panel but can be reselected.
DEFAULT_EXCLUDE_MICE = set()     # For example, {"m14", "m23"}

# Analysis settings
BASELINE_SECONDS = 5.0
EARLY_SECONDS = 7.0
LATE_SECONDS = 60.0
CHUNK_FRAMES = 48            # Number of frames read per chunk.
PLOT_BIN_SECONDS = 1.0       # For visualizing/exporting time courses only.
MASK_MODE = "brain"         # "brain" / "brain_signal"; the latter excludes bad/bregma/vessels masks.
LABEL_MOUSE_POINTS = True

# Optional disk cache of previously calculated traces across Jupyter sessions.
# Disabled by default to avoid stale results after regenerating Step4.
# The selection panel is sufficient for excluding mice; no cache is required.
USE_DISK_TRACE_CACHE = False
FORCE_RECOMPUTE_TRACES = False  # If disk caching is enabled, True forces trace recomputation.


In [ ]:
import hashlib
import json
import re
import warnings
import numpy as np
import pandas as pd
import yaml
import cv2
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, clear_output

plt.rcParams.update({"font.size": 10, "figure.dpi": 120,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"HbT": "#591d32", "HHb": "#173856", "HbO": "#c15b60"}
HB_SIGNALS = ("HbT", "HHb", "HbO")
SIGNALS = {name: "dc" + name + SIGNAL_SUFFIX for name in HB_SIGNALS}

assert 0 < BASELINE_SECONDS < EARLY_SECONDS and LATE_SECONDS > 0
assert PLOT_BIN_SECONDS > 0 and CHUNK_FRAMES > 0
assert MASK_MODE in {"brain", "brain_signal"}
assert SOURCE_HB_UNITS in {"M", "µM"}, "SOURCE_HB_UNITS должен быть 'M' или 'µM'"
HB_SCALE = 1e6 if SOURCE_HB_UNITS == "M" else 1.0
DISPLAY_UNITS = "µM"
print(f"Входные значения: {SOURCE_HB_UNITS}; множитель: {HB_SCALE:g}; выход: {DISPLAY_UNITS}")


## Locate the processed Zarr stores and masks

First try the analysis directory next to `CONFIG_PATH` (useful after migrating the project between drive letters). Only directories ending in `_processed.zarr` are selected. Separate `*_processed_motion_corrected.zarr` stores are not included.


In [ ]:
def locate_project(config_path, root_override=None):
    """Locate the existing analysis root and read its YAML configuration.

    Parameters
    ----------
    config_path : object
        Existing YAML config file path.
    root_override : object
        Optional explicit analysis root directory."""
    config_path = Path(config_path)
    cfg = yaml.safe_load(config_path.read_text(encoding="utf-8-sig"))
    project = cfg.get("project", {}) or {}
    if root_override is not None:
        candidates = [Path(root_override)]
    else:
        configured = str(project.get("directory_to_save_main_analysis", ""))
        leaf = configured.replace("\\", "/").rstrip("/").split("/")[-1]
        folder = project.get("analysis_folder_name") or leaf
        candidates = [config_path.parent / folder, Path(configured)]
        pr = cfg.get("path_resolving", {}) or {}
        old, new = pr.get("from_drive"), pr.get("to_drive")
        if configured and old and new and configured[:2].lower() == str(old)[:2].lower():
            candidates.insert(0, Path(str(new)[:2] + configured[2:]))
    for candidate in candidates:
        if candidate.is_dir():
            return candidate, cfg
    raise FileNotFoundError("Не найден корень анализа: " + "; ".join(map(str, candidates)))

ROOT, CFG = locate_project(CONFIG_PATH, PROJECT_ROOT_OVERRIDE)
OUT = ROOT / (CFG.get("project", {}) or {}).get("out_folder_name", "out")
STEP4 = CFG.get("step4", {}) or {}
STEP2 = CFG.get("step2", {}) or {}
PROCESSED = OUT / STEP4.get("folder_name", "step4_raw_and_signals") / STEP4.get("processed_signals_folder", "processed_signals")
MASKS = OUT / STEP2.get("folder_name", "step2_masks") / STEP2.get("masks_folder", "masks")
OUTPUT = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else PROCESSED.parent / "anest_start_summary"
FPS = float((CFG.get("raw", {}) or {}).get("fps", 0))
assert FPS > 0 and PROCESSED.is_dir() and MASKS.is_dir(), "Проверьте raw.fps и пути processed/masks"
STORES = sorted(p for p in PROCESSED.glob("*_processed.zarr") if p.is_dir() and ".incomplete" not in p.name)
assert STORES, f"Не найдено *_processed.zarr в {PROCESSED}"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Каталог Step4:", PROCESSED)
print("Маски Step2:", MASKS)
print("Zarr найдено:", len(STORES), "| FPS:", FPS)


In [ ]:
manifest_path = OUT / (CFG.get("step1", {}) or {}).get("manifest_name", "manifest_recordings.csv")
mouse_lookup = {}
if manifest_path.is_file():
    manifest = pd.read_csv(manifest_path)
    if {"stem", "mouse_id"}.issubset(manifest):
        mouse_lookup = dict(zip(manifest["stem"].astype(str), manifest["mouse_id"].astype(str)))

def canonical_mouse(x):
    """Normalize a mouse identifier for case-insensitive group selection.

    Parameters
    ----------
    x : object
        Input values."""
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def get_mouse_id(stem):
    """Look up the mouse identifier by stem, falling back to a stem-name pattern.

    Parameters
    ----------
    stem : object
        Recording stem identifier."""
    if stem in mouse_lookup:
        return str(mouse_lookup[stem])
    match = re.search(r"(?:^|_)(m_?\d+)(?=_|$)", str(stem), re.I)
    if match:
        return match.group(1)
    raise ValueError(f"Мышь не определена по stem {stem}; проверьте manifest")

def group_of(mouse, stem):
    """Assign a mouse or stem to exactly one configured group, if any.

    Parameters
    ----------
    mouse : object
        Mouse identifier.
    stem : object
        Recording stem identifier."""
    matched = [name for name, mice in GROUPS.items()
               if mice is None or any(canonical_mouse(item) == canonical_mouse(mouse) or str(item) == stem
                                   for item in mice)]
    if len(matched) > 1:
        raise ValueError(f"Мышь {mouse} попала в несколько групп: {matched}")
    return matched[0] if matched else None

SELECTION = pd.DataFrame([{
    "stem": store.name.removesuffix("_processed.zarr"),
    "mouse_id": get_mouse_id(store.name.removesuffix("_processed.zarr")),
    "path": store,
} for store in STORES])
print("Все записи для предварительного расчёта, включая потенциально исключаемых мышей:")
display(SELECTION.drop(columns="path"))


## Prepare temporal traces — run once

For each brain pixel, subtract its own mean from the first 5 seconds. Then calculate the brain-mask average for each time frame. Data are read in temporal chunks; the resulting **three time traces per mouse** and window summaries are held in memory. Conversion from M to µM occurs **after baseline subtraction** and does not modify the input Zarr.

**Important:** if Zarr metadata specify µM while `SOURCE_HB_UNITS='M'`, processing stops to prevent double scaling. Only when incorrect legacy metadata have been independently established should `ALLOW_METADATA_CONFLICT=True` be deliberately enabled, with the reason recorded in the analysis report.


In [ ]:
def load_brain_mask(stem, shape):
    """Load the Step2 brain mask, resize with nearest-neighbor interpolation, and optionally subtract exclusion masks.

    Parameters
    ----------
    stem : object
        Recording stem identifier.
    shape : object
        Target spatial shape, ``(y, x)``."""
    folder = MASKS / stem
    path = folder / "brain_manual.npy"
    if not path.is_file():
        path = folder / "brain_auto.npy"
    if not path.is_file():
        raise FileNotFoundError(f"Нет маски мозга для {stem} в {folder}")
    mask = np.load(path, allow_pickle=False) > 0
    if mask.ndim != 2:
        raise ValueError(f"Маска не двумерная: {path}")
    def resize(b):
        """Resize a binary mask to the processed signal grid using nearest-neighbor interpolation.

        Parameters
        ----------
        b : object
            Binary mask array."""
        return (cv2.resize(b.astype(np.uint8), (shape[1], shape[0]),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
                if b.shape != tuple(shape) else b.astype(bool))
    mask = resize(mask)
    if MASK_MODE == "brain_signal":
        for fname in ("bad_auto.npy", "bregma_manual.npy", "vessels_auto.npy"):
            other = folder / fname
            if fname == "bregma_manual.npy" and not other.is_file():
                other = folder / "bregma_auto.npy"
            if not other.is_file():
                raise FileNotFoundError(f"Не найдена маска исключения: {other}")
            mask &= ~resize(np.load(other, allow_pickle=False) > 0)
    if not mask.any():
        raise ValueError(f"Пустая маска: {stem}")
    return mask, path

def brain_trace_minus_first_seconds(da, mask, baseline_frames, chunk_frames=48):
    """Calculate a baseline-subtracted brain-averaged signal in temporal chunks.

    Parameters
    ----------
    da : xarray.DataArray
        Concentration movie with axes ``time, y, x`` (reordered if necessary).
    mask : numpy.ndarray of bool
        Two-dimensional spatial inclusion mask, indexed as ``[y, x]``.
    baseline_frames : int
        Number of initial frames used for each pixel's baseline.
    chunk_frames : int
        Number of frames processed in one read block.

    Returns
    -------
    trace : numpy.ndarray
        Per-frame spatial mean after pixelwise baseline subtraction, in input units.
    usable_pixel_count : int
        Pixels with a finite baseline and an included mask value.

    Notes
    -----
    The mask is applied in the processed signal grid. Finite samples are averaged
    per frame; NaNs do not contribute to that frame's spatial mean."""
    if set(da.dims) != {"time", "y", "x"}:
        raise ValueError(f"Ожидались time,y,x, получено {da.dims}")
    da = da.transpose("time", "y", "x")
    n, ny, nx = da.shape
    if mask.shape != (ny, nx):
        raise ValueError("Форма маски отличается от сетки processed")
    first = np.asarray(da.isel(time=slice(0, baseline_frames)).values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        baseline = np.nanmean(np.where(mask[None], first, np.nan), axis=0)
    usable = mask & np.isfinite(baseline)
    if not usable.any():
        raise ValueError("Нет пикселей с валидным baseline")
    trace = np.full(n, np.nan, dtype=np.float64)
    for a in range(0, n, chunk_frames):
        b = min(n, a + chunk_frames)
        chunk = np.asarray(da.isel(time=slice(a, b)).values, dtype=np.float64)
        delta = chunk - baseline[None, :, :]
        valid = usable[None, :, :] & np.isfinite(delta)
        summed = np.where(valid, delta, 0.0).sum(axis=(1, 2), dtype=np.float64)
        count = valid.sum(axis=(1, 2))
        trace[a:b] = np.divide(summed, count, out=np.full(b-a, np.nan), where=count>0)
    return trace, int(usable.sum())

def mean_window(x):
    """Return the mean of finite samples in one time window.

    Parameters
    ----------
    x : object
        Input values."""
    x = np.asarray(x, dtype=float)
    good = np.isfinite(x)
    return float(x[good].mean()) if good.any() else np.nan

def declared_unit(ds, varname):
    """Read the Hb concentration unit from a Zarr variable or dataset metadata.

    Parameters
    ----------
    ds : object
        Open Zarr dataset.
    varname : object
        Hb data variable name."""
    return str(ds[varname].attrs.get("units") or ds.attrs.get("hemoglobin_units") or "[not recorded]")

def canonical_unit(unit):
    """Normalize supported concentration-unit labels to M or µM.

    Parameters
    ----------
    unit : object
        Unit label."""
    u = str(unit).strip().lower().replace("μ", "µ").replace(" ", "")
    if u in {"m", "mol/l", "mol/liter", "mol/litre", "mol.l-1"}:
        return "M"
    if u in {"µm", "um", "µmol/l", "umol/l", "µmol.l-1"}:
        return "µM"
    return None

def validate_units(unit_by_signal, stem):
    """Reject declared Zarr units that conflict with the user-selected source units.

    Parameters
    ----------
    unit_by_signal : object
        Reported concentration units per Hb signal.
    stem : object
        Recording stem identifier."""
    for name, meta in unit_by_signal.items():
        canonical = canonical_unit(meta)
        if canonical is not None and canonical != SOURCE_HB_UNITS:
            msg = (f"{stem} / {name}: метаданные Zarr сообщают {meta}, "
                   f"но SOURCE_HB_UNITS={SOURCE_HB_UNITS}. Проверьте версию Step4.")
            if not ALLOW_METADATA_CONFLICT:
                raise ValueError(msg + " Для намеренного обхода установите ALLOW_METADATA_CONFLICT=True.")
            warnings.warn("ПРИНУДИТЕЛЬНОЕ ПРЕОБРАЗОВАНИЕ ПРИ КОНФЛИКТЕ: " + msg, RuntimeWarning)

def compute_stem(row):
    """Read one processed Zarr, calculate three Hb traces, and summarize early and late windows.

    Notes
    -----
    The notebook sets ``SOURCE_HB_UNITS`` explicitly and converts to µM via
    ``HB_SCALE`` after the pixelwise baseline subtraction and brain averaging.
    The input Step4 Zarr is read only and is closed before returning the record.
    The two analysis epochs are disjoint and the final epoch is relative to each
    recording's end."""
    ds = xr.open_zarr(row.path, consolidated=False, chunks=None, mask_and_scale=False)
    try:
        missing = [v for v in SIGNALS.values() if v not in ds]
        if missing:
            raise KeyError(f"{row.stem}: нет {missing}; доступны {list(ds.data_vars)}")
        shapes = {tuple(ds[v].transpose("time", "y", "x").shape) for v in SIGNALS.values()}
        if len(shapes) != 1:
            raise ValueError(f"У сигналов разные формы: {shapes}")
        n, ny, nx = shapes.pop()
        nb = int(round(BASELINE_SECONDS * FPS))
        ne = int(round(EARLY_SECONDS * FPS))
        nl = int(round(LATE_SECONDS * FPS))
        if not (nb >= 1 and nb < ne and ne >= 1 and nl >= 1 and n >= ne+nl):
            raise ValueError(f"{row.stem}: недостаточно кадров ({n}) для непересекающихся окон {EARLY_SECONDS}с и {LATE_SECONDS}с")
        units = {name: declared_unit(ds, varname) for name, varname in SIGNALS.items()}
        validate_units(units, row.stem)
        mask, mask_path = load_brain_mask(row.stem, (ny, nx))
        traces, pixels, summary = {}, {}, {}
        for name, varname in SIGNALS.items():
            trace, pix = brain_trace_minus_first_seconds(ds[varname], mask, nb, CHUNK_FRAMES)
            trace = trace * HB_SCALE
            traces[name] = trace
            pixels[name] = pix
            early, late = mean_window(trace[:ne]), mean_window(trace[-nl:])
            summary[name] = {"early": early, "late": late, "delta": late-early, "pixels": pix, "unit": DISPLAY_UNITS}
        return {"stem": str(row.stem), "mouse_id": str(row.mouse_id), "mask_file": str(mask_path),
                "processed_path": str(row.path), "time": np.arange(n, dtype=float)/FPS,
                "traces": traces, "summary": summary, "stored_units": units,
                "duration_sec": n/FPS, "brain_pixels": int(mask.sum())}
    finally:
        ds.close()


### Compute traces for all selected Zarr stores

This cell reads the input data. **Do not rerun it merely to exclude a mouse**; use the selection panel below. The optional disk cache stores only derived traces and settings and is disabled by default to prevent stale results after Step4 Zarr stores are regenerated.


In [ ]:
def cache_filename(row):
    """Derive a fingerprinted output path for one derived trace-cache file.

    Parameters
    ----------
    row : object
        Metadata record for one recording."""
    key = {"path": str(row.path), "stem": str(row.stem), "variant": SIGNAL_SUFFIX,
           "fps": FPS, "baseline": BASELINE_SECONDS, "early": EARLY_SECONDS,
           "late": LATE_SECONDS, "mask": MASK_MODE, "source_units": SOURCE_HB_UNITS,
           "scale": HB_SCALE, "version": 2}
    token = hashlib.sha256(json.dumps(key, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    return OUTPUT / "trace_cache" / f"{row.stem}_{token}.npz"

def save_trace_cache(path, record):
    """Write derived time arrays, traces, and metadata into a compressed NPZ cache.

    Parameters
    ----------
    path : object
        File path.
    record : object
        Derived traces and metadata."""
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {k: v for k, v in record.items() if k not in {"time", "traces"}}
    np.savez_compressed(path, metadata_json=json.dumps(metadata, ensure_ascii=False),
                        time=record["time"], **{f"trace_{k}": v for k, v in record["traces"].items()})

def load_trace_cache(path):
    """Restore derived traces and metadata from the compressed NPZ cache.

    Parameters
    ----------
    path : object
        File path."""
    with np.load(path, allow_pickle=False) as saved:
        record = json.loads(str(saved["metadata_json"].item()))
        record["time"] = saved["time"].copy()
        record["traces"] = {k: saved[f"trace_{k}"].copy() for k in HB_SIGNALS}
    return record

records = {}
failed = []
for i, row in enumerate(SELECTION.itertuples(index=False), 1):
    try:
        cache_path = cache_filename(row)
        if USE_DISK_TRACE_CACHE and cache_path.exists() and not FORCE_RECOMPUTE_TRACES:
            record = load_trace_cache(cache_path)
            source = "disk cache"
        else:
            record = compute_stem(row)
            source = "processed Zarr"
            if USE_DISK_TRACE_CACHE:
                save_trace_cache(cache_path, record)
        records[record["stem"]] = record
        print(f"[{i}/{len(SELECTION)}] OK {record['stem']} | {source} | {record['brain_pixels']} brain pixels")
    except Exception as exc:
        failed.append({"stem": row.stem, "error": repr(exc)})
        print(f"[{i}/{len(SELECTION)}] ERROR {row.stem}: {exc}")
if not records:
    raise RuntimeError("Ни одна запись не обработана; смотрите ошибки выше")
if failed:
    display(pd.DataFrame(failed))
unit_table = pd.DataFrame([{"stem": r["stem"], "mouse_id": r["mouse_id"],
                            **{f"zarr_{k}_units": v for k, v in r["stored_units"].items()},
                            "assumed_input_units": SOURCE_HB_UNITS, "output_units": DISPLAY_UNITS,
                            "scale_factor": HB_SCALE} for r in records.values()])
display(unit_table)
print("Готово: временные ряды", len(records), "мышей сохранены в памяти. Далее Zarr НЕ перечитываются.")
if USE_DISK_TRACE_CACHE:
    print("ВНИМАНИЕ: если processed Zarr или маски будут перезаписаны, включите FORCE_RECOMPUTE_TRACES=True или удалите trace_cache.")


## Statistics and figures

Everything below operates **only on `records`**, which contains already computed one-dimensional traces. Clicking the panel's refresh button regenerates tables, tests, scatter and boxplots, and time-course plots without rereading the Zarr stores or masks.

Statistics are computed on **individual mouse-level Δ values**. Time courses are restricted to the duration shared by the included mice; group mean and SD are calculated **across mice**. The final 60-second window is defined relative to the end of each individual recording, even if recording durations differ slightly.


In [ ]:
def holm_adjust(pvalues):
    """Apply the Holm family-wise error-rate adjustment to finite p-values.

    Parameters
    ----------
    pvalues : object
        Unadjusted p-values."""
    pvalues = np.asarray(pvalues, dtype=float)
    answer = np.full(len(pvalues), np.nan, dtype=float)
    ids = np.flatnonzero(np.isfinite(pvalues))
    ids = ids[np.argsort(pvalues[ids])]
    running = 0.0
    for k, idx in enumerate(ids):
        running = max(running, float(len(ids)-k) * pvalues[idx])
        answer[idx] = min(1.0, running)
    return answer

def stats_against_zero(values):
    """Compute mouse-level summary statistics and tests for a vector of Hb deltas.

    Notes
    -----
    Uses a two-sided one-sample t-test of individual mouse deltas against zero.
    Wilcoxon is included as an exploratory result; Holm correction is applied
    later across the three Hb components in each group."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    mu = float(np.mean(x)) if n else np.nan
    sd = float(np.std(x, ddof=1)) if n > 1 else np.nan
    sem = sd / np.sqrt(n) if n > 1 else np.nan
    if n > 1 and sd > 0:
        t, p = stats.ttest_1samp(x, 0.0, alternative="two-sided")
        lo, hi = stats.t.interval(0.95, n-1, loc=mu, scale=sem)
    else:
        t, p, lo, hi = np.nan, np.nan, np.nan, np.nan
    try:
        p_w = float(stats.wilcoxon(x, alternative="two-sided").pvalue) if n > 1 and np.any(x != 0) else np.nan
    except ValueError:
        p_w = np.nan
    return {"n": n, "mean_delta": mu, "SD": sd, "SEM": sem, "CI95_low": lo,
            "CI95_high": hi, "t_statistic": t, "p_t": p, "p_wilcoxon_exploratory": p_w}

def build_analysis_tables(allowed_mouse_ids):
    """Build full, included, and group-level tables from already loaded mouse traces.

    Notes
    -----
    The function refuses multiple included recordings of the same mouse within
    one group; the statistical sample size must count independent mice."""
    allowed = {canonical_mouse(mouse) for mouse in allowed_mouse_ids}
    rows = []
    for stem, r in records.items():
        group = group_of(r["mouse_id"], stem)
        included = group is not None and canonical_mouse(r["mouse_id"]) in allowed
        for name in HB_SIGNALS:
            v = r["summary"][name]
            rows.append({"group": group, "mouse_id": r["mouse_id"], "stem": stem,
                         "included": included, "signal": name,
                         "early_7s": v["early"], "late_60s": v["late"],
                         "delta_late_minus_early": v["delta"], "pixels_with_baseline": v["pixels"],
                         "units": DISPLAY_UNITS, "duration_sec": r["duration_sec"]})
    full = pd.DataFrame(rows)
    included = full[full["included"]].copy()
    if included.empty:
        raise ValueError("Все мыши выключены или не включены ни в одну группу")
    duplicates = included[["group", "mouse_id", "stem"]].drop_duplicates().duplicated(["group", "mouse_id"])
    if duplicates.any():
        raise ValueError("В группе несколько записей одной мыши. Нельзя считать их независимыми n без агрегации.")
    group_stat_rows = []
    for (group, name), part in included.groupby(["group", "signal"], sort=False):
        result = stats_against_zero(part["delta_late_minus_early"])
        result.update(group=group, signal=name, units=DISPLAY_UNITS,
                      mean_plusminus_SD=f"{result['mean_delta']:.5g} ± {result['SD']:.5g}")
        group_stat_rows.append(result)
    grouped = pd.DataFrame(group_stat_rows)
    grouped["p_holm_3Hb"] = np.nan
    for group in grouped.group.unique():
        ix = grouped.index[grouped.group.eq(group)]
        grouped.loc[ix, "p_holm_3Hb"] = holm_adjust(grouped.loc[ix, "p_t"])
    return full, included, grouped

def binned_trace(time, trace, bin_seconds):
    """Average finite trace samples into consecutive time bins.

    Parameters
    ----------
    time : object
        Sample timestamps in seconds.
    trace : object
        Hemoglobin time series.
    bin_seconds : object
        Time-bin width in seconds."""
    time, trace = np.asarray(time, float), np.asarray(trace, float)
    idx = np.floor(time / bin_seconds).astype(int)
    good = np.isfinite(trace)
    n = int(idx[-1] + 1)
    sums = np.bincount(idx[good], weights=trace[good], minlength=n)
    count = np.bincount(idx[good], minlength=n)
    return np.divide(sums, count, out=np.full(n, np.nan, dtype=float), where=count > 0)


In [ ]:
def save_show_figure(fig, name, show=True):
    """Save a Matplotlib figure, optionally display it, and close it.

    Parameters
    ----------
    fig : object
        Matplotlib figure object.
    name : object
        Output filename.
    show : object
        Whether to display the figure."""
    fig.savefig(OUTPUT / name, dpi=220, bbox_inches="tight")
    if show:
        display(fig)
    plt.close(fig)

def plot_delta_scatter_box(included, grouped, show=True):
    """Plot each included mouse delta and the per-group Hb-component boxplots.

    Parameters
    ----------
    included : object
        Selected mouse-level table.
    grouped : object
        Group-level statistics table.
    show : object
        Whether to display the figure."""
    groups = [g for g in GROUPS if included.group.eq(g).any()]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.7), constrained_layout=True)
    rng = np.random.default_rng(2026)
    for ax, name in zip(axes, HB_SIGNALS):
        color = COLORS[name]
        for j, group in enumerate(groups, 1):
            sub = included[(included.group == group) & (included.signal == name)]
            sub = sub[np.isfinite(sub.delta_late_minus_early)]
            values = sub.delta_late_minus_early.to_numpy(dtype=float)
            if len(values):
                ax.boxplot(values, positions=[j], widths=0.45, patch_artist=True, showfliers=False,
                           boxprops={"facecolor": color, "edgecolor": color, "alpha": .17},
                           medianprops={"color": color, "linewidth": 2},
                           whiskerprops={"color": color}, capprops={"color": color})
                jitter = rng.uniform(-0.13, 0.13, len(values))
                ax.scatter(j + jitter, values, c=color, s=41, alpha=.9,
                           edgecolors="white", linewidths=.6, zorder=4)
                if LABEL_MOUSE_POINTS:
                    for xx, y, mouse in zip(j+jitter, values, sub.mouse_id):
                        ax.annotate(str(mouse), (xx, y), xytext=(4, 4),
                                    textcoords="offset points", fontsize=7, color=color)
            st = grouped[(grouped.group == group) & (grouped.signal == name)]
            p = float(st.iloc[0]["p_holm_3Hb"]) if len(st) else np.nan
            label = f"n={len(values)}, p_Holm={p:.3g}" if np.isfinite(p) else f"n={len(values)}, p=n/a"
            ax.text(j, 1.04, label, transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=8)
        ax.axhline(0, color="0.35", ls="--", lw=1)
        ax.set_xticks(range(1, len(groups)+1), groups)
        ax.set_xlim(0.4, len(groups) + 0.6)
        ax.set_ylabel(f"Δ (последние {LATE_SECONDS:g} − первые {EARLY_SECONDS:g} с), {DISPLAY_UNITS}")
        ax.set_title(name, color=color, fontweight="bold", pad=28)
        ax.grid(axis="y", alpha=.2)
    save_show_figure(fig, "hb_delta_scatter_boxplot.png", show=show)

def curves_for_group(group, included):
    """Prepare individual and group-level trace data for one selected group.

    Parameters
    ----------
    group : object
        Group label.
    included : object
        Selected mouse-level table."""
    selected_stems = included.loc[included.group.eq(group), "stem"].unique().tolist()
    chosen = [records[stem] for stem in selected_stems]
    if not chosen:
        return {}, pd.DataFrame(), pd.DataFrame()
    n_common = min(len(binned_trace(r["time"], r["traces"]["HbT"], PLOT_BIN_SECONDS)) for r in chosen)
    t = (np.arange(n_common) + 0.5) * PLOT_BIN_SECONDS
    curves, group_rows, individual_rows = {}, [], []
    for name in HB_SIGNALS:
        lines = []
        for r in chosen:
            b = binned_trace(r["time"], r["traces"][name], PLOT_BIN_SECONDS)
            lines.append(b[:n_common])
            for ix, v in enumerate(b):
                individual_rows.append({"group": group, "mouse_id": r["mouse_id"],
                                        "stem": r["stem"], "signal": name,
                                        "t_sec": (ix+0.5)*PLOT_BIN_SECONDS, "mean_signal": v})
        matrix = np.stack(lines)
        valid = np.isfinite(matrix)
        n = valid.sum(axis=0)
        means = np.divide(np.where(valid, matrix, 0).sum(axis=0), n,
                          out=np.full(n_common, np.nan), where=n > 0)
        deviations = np.where(valid, matrix - means, 0.0)
        sds = np.sqrt(np.divide((deviations**2).sum(axis=0), n-1,
                                out=np.full(n_common, np.nan), where=n > 1))
        curves[name] = {"time": t, "mean": means, "SD": sds, "individual": matrix,
                        "mouse_ids": [r["mouse_id"] for r in chosen]}
        for tt, mean, sd, nn in zip(t, means, sds, n):
            group_rows.append({"group": group, "signal": name, "t_sec": tt,
                               "mean": mean, "SD": sd, "n_mice": nn})
    return curves, pd.DataFrame(group_rows), pd.DataFrame(individual_rows)

def shade_time_windows(ax, chosen, max_time):
    """Mark the early and late analysis windows on a time-course axis.

    Parameters
    ----------
    ax : object
        Matplotlib axes.
    chosen : object
        Selected mouse records.
    max_time : object
        Maximum displayed time in seconds."""
    ax.axvspan(0, EARLY_SECONDS, color="0.6", alpha=.11, label="Первые 7 с")
    durations = np.asarray([r["duration_sec"] for r in chosen])
    if len(durations) and np.ptp(durations) < PLOT_BIN_SECONDS * .5:
        ax.axvspan(max(0, durations[0] - LATE_SECONDS), min(max_time, durations[0]),
                   color="0.75", alpha=.12, label="Последние 60 с")

def plot_time_courses(group, group_index, included, show=True):
    """Plot group and individual time courses with between-mouse variability.

    Parameters
    ----------
    group : object
        Group label.
    group_index : object
        Order of the group in plotted panels.
    included : object
        Selected mouse-level table.
    show : object
        Whether to display the figure."""
    curves, group_table, individual_table = curves_for_group(group, included)
    if not curves:
        return group_table, individual_table
    selected_stems = included.loc[included.group.eq(group), "stem"].unique().tolist()
    chosen = [records[stem] for stem in selected_stems]
    end = float(curves["HbT"]["time"][-1] + PLOT_BIN_SECONDS/2)
    fig, ax = plt.subplots(figsize=(13.4, 5.5), constrained_layout=True)
    for name in HB_SIGNALS:
        c = curves[name]
        ax.plot(c["time"], c["mean"], lw=2, color=COLORS[name], label=f"{name}, mean")
        if len(chosen) > 1:
            ax.fill_between(c["time"], c["mean"]-c["SD"], c["mean"]+c["SD"],
                            color=COLORS[name], alpha=.16, linewidth=0)
    shade_time_windows(ax, chosen, end)
    ax.axhline(0, color="0.35", lw=.9, ls="--")
    ax.set(xlabel="Время (с)", ylabel=f"ΔHb от baseline первых 5 с ({DISPLAY_UNITS})",
           title=f"{group}: средние по мышам ± SD (n={len(chosen)})")
    ax.legend(ncol=5, fontsize=9)
    ax.grid(alpha=.18)
    save_show_figure(fig, f"group_{group_index:02d}_mean_Hb_timecourse.png", show=show)

    fig, axes = plt.subplots(3, 1, figsize=(13.4, 10.4), sharex=True, constrained_layout=True)
    for ax, name in zip(axes, HB_SIGNALS):
        c = curves[name]
        for mouse, line in zip(c["mouse_ids"], c["individual"]):
            ax.plot(c["time"], line, color=COLORS[name], alpha=.29, lw=1,
                    label=str(mouse) if name == "HbT" else None)
        ax.plot(c["time"], c["mean"], color=COLORS[name], lw=2.7, label="Среднее группы")
        if len(chosen) > 1:
            ax.fill_between(c["time"], c["mean"]-c["SD"], c["mean"]+c["SD"],
                            color=COLORS[name], alpha=.12)
        shade_time_windows(ax, chosen, end)
        ax.axhline(0, color="0.4", lw=.9, ls="--")
        ax.set_ylabel(f"{name} ({DISPLAY_UNITS})")
        ax.grid(alpha=.18)
    axes[-1].set_xlabel("Время от начала записи (с)")
    axes[0].set_title(f"{group}: отдельные мыши (тонкие линии) + mean ± SD", fontsize=12)
    if len(chosen) <= 15:
        axes[0].legend(loc="best", ncol=min(6, len(chosen)+2), fontsize=8)
    save_show_figure(fig, f"group_{group_index:02d}_individual_Hb_timecourses.png", show=show)
    return group_table, individual_table


## Mouse-selection panel and analysis refresh

Select or clear the mouse checkboxes, then press the refresh-analysis button. No `processed.zarr` stores are reopened; the calculations use `records`.

* `all_mice` contains **all** successfully loaded recordings, including the `included` flag.
* `included_mouse` and `group_stats` contain only the **currently selected mice**.
* Selection controls statistics, scatter and boxplots, group and individual time courses, and exported CSV/PNG files.
* If `ipywidgets` is unavailable, call `run_analysis(...)` directly in the same code cell. To change the group definitions, edit `GROUPS` in the settings and rerun the relevant selection logic.

**The button and widget labels are still in Russian:** these are runtime string literals and have intentionally not been changed in this documentation-only revision.


In [ ]:
def run_analysis(included_mouse_ids=None, show_plots=True, show_tables=True):
    """Recompute statistics and figures from in-memory traces without rereading Zarr."""
    if included_mouse_ids is None:
        defaults = {canonical_mouse(x) for x in DEFAULT_EXCLUDE_MICE}
        included_mouse_ids = [r["mouse_id"] for r in records.values()
                              if canonical_mouse(r["mouse_id"]) not in defaults
                              and r["stem"] not in DEFAULT_EXCLUDE_MICE]
    full, included, grouped = build_analysis_tables(included_mouse_ids)
    group_tables, individual_tables = [], []
    plot_delta_scatter_box(included, grouped, show=show_plots)
    for index, group in enumerate(GROUPS, 1):
        gt, it = plot_time_courses(group, index, included, show=show_plots)
        if not gt.empty:
            group_tables.append(gt)
        if not it.empty:
            individual_tables.append(it)
    group_traces = (pd.concat(group_tables, ignore_index=True) if group_tables else
                    pd.DataFrame(columns=["group", "signal", "t_sec", "mean", "SD", "n_mice"]))
    individual_binned = (pd.concat(individual_tables, ignore_index=True) if individual_tables else
                         pd.DataFrame(columns=["group", "mouse_id", "stem", "signal", "t_sec", "mean_signal"]))
    OUTPUT.mkdir(parents=True, exist_ok=True)
    full.to_csv(OUTPUT / "all_mice_7s_60s_including_excluded.csv", index=False, encoding="utf-8-sig")
    included.to_csv(OUTPUT / "included_mice_7s_60s.csv", index=False, encoding="utf-8-sig")
    grouped.to_csv(OUTPUT / "group_delta_statistics.csv", index=False, encoding="utf-8-sig")
    unit_table.to_csv(OUTPUT / "units_by_stem.csv", index=False, encoding="utf-8-sig")
    group_traces.to_csv(OUTPUT / "group_mean_traces_binned.csv", index=False, encoding="utf-8-sig")
    individual_binned.to_csv(OUTPUT / "individual_mouse_traces_binned.csv", index=False, encoding="utf-8-sig")
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT / "failed_stems.csv", index=False, encoding="utf-8-sig")
    included_animals = full[full.included]["mouse_id"].unique().tolist()
    excluded_animals = full[~full.included]["mouse_id"].unique().tolist()
    params = {"config_path": str(CONFIG_PATH), "project_root": str(ROOT),
              "input_units_assumed": SOURCE_HB_UNITS, "conversion_to_uM": HB_SCALE,
              "metadata_conflict_override": ALLOW_METADATA_CONFLICT,
              "output_units": DISPLAY_UNITS, "signal_suffix": SIGNAL_SUFFIX,
              "groups": GROUPS, "included_mice": included_animals, "excluded_mice": excluded_animals,
              "baseline_seconds": BASELINE_SECONDS, "early_seconds": EARLY_SECONDS,
              "late_seconds": LATE_SECONDS, "fps": FPS, "mask_mode": MASK_MODE,
              "plot_bin_seconds": PLOT_BIN_SECONDS,
              "statistics": "2-sided one-sample t-test mouse deltas vs 0; Holm across HbT, HHb, HbO per group",
              "n_stems_loaded": len(records), "failed_stems": failed}
    (OUTPUT / "analysis_parameters.json").write_text(
        json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
    if show_tables:
        print("Включено мышей:", included[["group", "mouse_id"]].drop_duplicates().groupby("group").size().to_dict())
        display(included.sort_values(["group", "mouse_id", "signal"]))
        display(grouped[["group", "signal", "n", "mean_plusminus_SD", "CI95_low", "CI95_high",
                         "p_t", "p_holm_3Hb", "p_wilcoxon_exploratory", "units"]].round(5))
        print("Готовые CSV/PNG:", OUTPUT)
    # Reuse/export results in subsequent cells without processing Zarr again.
    result = {"all_mice": full, "included_mouse": included, "group_stats": grouped,
              "group_traces": group_traces, "individual_binned": individual_binned}
    globals().update(result)
    return result

try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets не установлен. Для галочек: pip install ipywidgets")
    print("Без widgets можно вызвать run_analysis(included_mouse_ids=['m1', 'm14', ...])")
    analysis_result = run_analysis()
else:
    mice = sorted({r["mouse_id"] for r in records.values()},
                  key=lambda x: (int(re.search(r"\d+", x).group()) if re.search(r"\d+", x) else 9999, x))
    defaults = {canonical_mouse(x) for x in DEFAULT_EXCLUDE_MICE}
    checkboxes = {
        mouse: widgets.Checkbox(value=(canonical_mouse(mouse) not in defaults),
                                description=mouse, indent=False, layout=widgets.Layout(width="125px"))
        for mouse in mice
    }
    select_all_btn = widgets.Button(description="Включить всех", icon="check")
    select_none_btn = widgets.Button(description="Снять все", icon="times")
    refresh_btn = widgets.Button(description="Обновить статистику и графики",
                                 button_style="primary", icon="refresh",
                                 layout=widgets.Layout(width="285px"))
    info_label = widgets.HTML("<b>Выберите мышей для анализа (изменяются только графики и статистика):</b>")
    output_widget = widgets.Output()
    def set_all(yes):
        """Set all interactive mouse-selection checkboxes to the same state.

        Parameters
        ----------
        yes : object
            Checkbox state."""
        for cb in checkboxes.values():
            cb.value = yes
    select_all_btn.on_click(lambda _: set_all(True))
    select_none_btn.on_click(lambda _: set_all(False))
    def refresh_analysis(_=None):
        """Recalculate the analysis for mice currently selected in the widget panel.

        Parameters
        ----------
        _ : object
            Event object (not used)."""
        chosen = [mouse for mouse, cb in checkboxes.items() if cb.value]
        with output_widget:
            clear_output(wait=True)
            if not chosen:
                print("Не выбрана ни одна мышь. Отметьте хотя бы одну и обновите.")
                return
            try:
                globals()["analysis_result"] = run_analysis(chosen)
            except Exception as exc:
                import traceback
                traceback.print_exc()
    refresh_btn.on_click(refresh_analysis)
    grid = widgets.GridBox(list(checkboxes.values()),
                           layout=widgets.Layout(grid_template_columns="repeat(5, 125px)", grid_gap="3px"))
    panel = widgets.VBox([info_label, grid,
                          widgets.HBox([select_all_btn, select_none_btn, refresh_btn]), output_widget])
    display(panel)
    refresh_analysis()


## Interpretation notes

- A constant baseline subtraction cancels out of Δ calculated from the same trace; subtracting each pixel's first-5-second baseline remains part of the stated trace-processing procedure for comparability.
- In plots, **SD means variability across mice**—not SEM or variability across brain pixels. `n` is the number of mice.
- For small n, the t-test is sensitive to outliers and distributional assumptions. Wilcoxon is an exploratory check, not a replacement for the primary test. Mouse exclusions should be documented before interpreting the results.
- When recordings differ in length, the last 60 seconds are selected **from each mouse's own recording end**; time-course plots include only the common starting time range. The late-window shading appears only when durations are sufficiently similar.
- After **regenerating processed Zarr stores**, rerun the data-preparation cell. If the disk cache is enabled, force recomputation or invalidate the stored cache.
